# Berlin property market overview

Starter analysis on the `listing_features` view. Run `immo crawl` and `immo enrich` first.

Start with: `uv run --extra analysis jupyter lab`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np  # used inside model formulas
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

from immo_scanner.explore.data import load_features, load_price_history

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

all_listings = load_features(active_only=False)
active = all_listings[all_listings["status"] == "active"].copy()
print(f"{len(active)} active / {len(all_listings)} total listings")

## Coverage by source

In [ ]:
all_listings.groupby(["source", "status"]).agg(
    listings=("id", "count"),
    with_location=("lat", "count"),
    median_price=("price", "median"),
    median_eur_m2=("price_per_m2", "median"),
).round(0)

## Prices by Bezirk

In [ ]:
by_bezirk = (
    active.dropna(subset=["bezirk", "price_per_m2"])
    .groupby("bezirk")["price_per_m2"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .sort_values("50%", ascending=False)
    .round(0)
)
by_bezirk

In [ ]:
plot_df = active.dropna(subset=["bezirk", "price_per_m2"])
plot_df = plot_df[plot_df["price_per_m2"].between(1000, 20000)]
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=plot_df, y="bezirk", x="price_per_m2", order=by_bezirk.index, ax=ax)
ax.set(xlabel="EUR per m²", ylabel="")
plt.show()

## Location factors

How do commute time and distance to rail relate to price per m²?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=plot_df, x="commute_minutes", y="price_per_m2", hue="source", alpha=0.5, ax=axes[0])
sns.scatterplot(data=plot_df, x="nearest_rail_m", y="price_per_m2", hue="source", alpha=0.5, ax=axes[1])
axes[1].set_xlim(0, 3000)
plt.show()

## Hedonic price model

A simple linear model of price per m². Listings far *below* the prediction (negative residual) are worth a closer look.

In [ ]:
model_df = active.dropna(
    subset=["price_per_m2", "living_area", "rooms", "nearest_rail_m", "bezirk"]
).copy()
model_df = model_df[model_df["price_per_m2"].between(1000, 20000)]
model_df["is_new_build"] = model_df["is_new_build"].astype(int)

formula = "price_per_m2 ~ living_area + rooms + np.log1p(nearest_rail_m) + is_new_build + C(bezirk)"
if model_df["commute_minutes"].notna().mean() > 0.8:
    model_df = model_df.dropna(subset=["commute_minutes"])
    formula += " + commute_minutes"

model = smf.ols(formula, data=model_df).fit()
print(model.summary().tables[1])
print(f"R² = {model.rsquared:.2f}, n = {int(model.nobs)}")

In [ ]:
model_df["predicted_eur_m2"] = model.fittedvalues.round(0)
model_df["residual_pct"] = (model.resid / model.fittedvalues * 100).round(1)
model_df.sort_values("residual_pct")[
    ["title", "bezirk", "price", "living_area", "price_per_m2", "predicted_eur_m2",
     "residual_pct", "commute_minutes", "nearest_rail_m", "url"]
].head(25)

## Market dynamics: price changes and time on market

In [ ]:
removed = all_listings[all_listings["status"] == "removed"]
print("Median days on market (removed listings):", removed["days_on_market"].median())
print("Share of active listings with a price drop:",
      round((active["price_change_pct"] < 0).mean() * 100, 1), "%")

active[active["price_change_pct"] < 0].sort_values("price_change_pct")[
    ["title", "bezirk", "first_price", "price", "price_change_pct", "days_on_market", "url"]
].head(20)

In [ ]:
history = load_price_history()
weekly = (
    history.assign(week=history["scraped_at"].dt.to_period("W").dt.start_time,
                   eur_m2=history["price"] / history["living_area"])
    .groupby(["week", "source"])["eur_m2"].median()
    .unstack()
)
weekly.plot(figsize=(10, 4), title="Median asking price per m² of snapshots by week")
plt.show()

## Neighbourhood context: demographics and election results

In [ ]:
context_cols = ["price_per_m2", "share_under_18_pct", "share_65_plus_pct", "agh_turnout_pct",
                "agh_vote_linke", "agh_vote_spd", "agh_vote_gruene", "agh_vote_cdu",
                "agh_vote_afd", "agh_vote_fdp", "agh_vote_bsw"]
corr = active[[c for c in context_cols if c in active]].corr(numeric_only=True)["price_per_m2"]
corr.drop("price_per_m2").sort_values().plot.barh(figsize=(7, 4), title="Correlation with EUR/m²")
plt.show()